### Installiamo `gurobipy`, documentazione @ https://www.gurobi.com/documentation/9.5/refman/py_python_api_details.html#sec:Python-details

In [47]:
!pip3 install gurobipy
!pip3 install numpy
!pip3 install scipy

# Importiamo i pacchetti necessari

In [48]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Volgiamo definire e risolvere il seguente problema

\begin{align}
  \text{maximize}\ &250x_1 + 230x_2 + 110x_3 + 350x_4 + 110x_5\\
                  &s.t.\\
                  &0\le\ x_i \lt \infty\\
                  &2x_1 + 1.5x_2 + 0.5x_3 + 2.5x_4 + 0.7x_5\leq 100.\\
                  &0.5x_1 + 0.25x_2 + 0.25x_3 + x_4 + 0.3x_5\leq 50\\
                  &x_1 + x_2 \geq 10\\
                  &x_3 = x_5\\
\end{align}

Creiamo il modello

In [49]:
model = gp.Model()

Aggiungiamo le variabili con i rispettivi limiti

In [50]:
n_vars = 5
x = model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
model.update()

In [51]:
print(f"Model vars:{model.getVars()}\nNum vars: {len(model.getVars())}\nVar C1: {model.getVarByName('C0')}")

Model vars:[<gurobi.Var C0>, <gurobi.Var C1>, <gurobi.Var C2>, <gurobi.Var C3>, <gurobi.Var C4>]
Num vars: 5
Var C1: <gurobi.Var C0>


Aggiungiamo i vincoli del problema

In [52]:
A =  np.array([[2., 1.5, 0.5, 2.5, 0.7],

               [0.5, 0.25, 0.25, 1, 0.3],
               [1, 1, 0, 0, 0],
               [0, 0, 1, 0, -1]])

b = np.array([100, 50, 10, 0])
ct = model.addConstr(A[:2]@x <= b[:2])
ct2 = model.addConstr(A[-2]@x >= b[-2])
ct3 = model.addConstr(A[-1]@x == b[-1])

model.update()


Definiamo la funzione obiettivo

In [53]:
obj_coefs = np.array([250, 230, 110, 350, 110])
model.setObjective(obj_coefs @ x, GRB.MAXIMIZE)

Risolviamo il problema

In [54]:
model.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E246)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 4 rows, 5 columns and 14 nonzeros (Max)
Model fingerprint: 0x98c1760a
Model has 5 linear objective coefficients
Coefficient statistics:
  Matrix range     [2e-01, 2e+00]
  Objective range  [1e+02, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+01, 1e+02]

Presolve removed 1 rows and 1 columns
Presolve time: 0.00s
Presolved: 3 rows, 4 columns, 10 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.8333333e+04   2.500000e+00   0.000000e+00      0s
       1    1.7883333e+04   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.788333333e+04


Stampiamo per ogni variabile il valore nella soluzione ottima



In [55]:
for v in model.getVars():
  print('%s %g' % (v.VarName, v.X))

print('Obj: %g' % model.ObjVal)

C0 0
C1 10
C2 70.8333
C3 0
C4 70.8333
Obj: 17883.3


# Proviamo a risolvere il problema duale

In [ ]:
#TODO
dual_model = gp.Model()
n_vars = 4
x = dual_model.addMVar(n_vars, lb=[0] * n_vars, ub=GRB.INFINITY)
dual_model.update()

c = np.array([100, 50, 10, 0], dtype=float)
A = np.array([
    [2.0,   0.5,  1.0,  0.0],   # 2y1 + 0.5y2 + 1y3       >= 250
    [1.5,   0.25, 1.0,  0.0],   # 1.5y1 + 0.25y2 + 1y3    >= 230
    [0.5,   0.25, 0.0,  1.0],   # 0.5y1 + 0.25y2 + 1y4    >= 110
    [2.5,   1.0,  0.0,  0.0],   # 2.5y1 + 1y2             >= 350
    [0.7,   0.3,  0.0, -1.0]    # 0.7y1 + 0.3y2 - 1y4     >= 110
], dtype=float)

b = np.array([250, 230, 110, 350, 110], dtype=float)



# E' possibile accedere ai vincoli ed alla funzione obiettivo

In [57]:
for c in dual_model.getConstrs():
  print(dual_model.getRow(c))

In [58]:
dual_model.getObjective()

<gurobi.LinExpr: 0.0>

# Rendiamo il problema primale privo di soluzioni ammissibili

# Rendiamo il problema primale illimitato

In [59]:
#TODO

# Problema del cammino minimo
Dato un grafo $\mathcal{G}$ e due nodi $s,\ t$ vogliamo trovare il cammino che minimiza il costo di andare da $s$ a $t$.

Creiamo un grafo costituito da $200$ nodi

In [60]:
n_nodes = 200
adj_mat = np.random.rand(n_nodes, n_nodes)
adj_mat[adj_mat > 0.02] = 0
adj_mat *= 1000
adj_mat = np.round(adj_mat)
np.fill_diagonal(adj_mat, 0)

Definiamo il nodo di partenza ed il nodo di arrivo

In [61]:
s = 0
t = 197

## Risolviamo il problema con Gurobi

Se il costo di ogni arco e' positivo, il problema si puo' formulare nel seguente modo:
\begin{align}
  \text{minimize} &\sum_{ij} c_{ij}x_{ij}\\
                  & s.t.\\
                  0 \leq\ &x \leq 1\\
                  \sum_i x_{iv} + b_v &= \sum_j x_{vj},\ \forall v \in \text{Nodes}
\end{align}
dove $b_v$ il flusso in ingresso su ogni nodo ed $x_{ij}$ la quantita' di flusso nell'arco $(i, j)$.

In [62]:
import gurobipy as gp
from gurobipy import GRB

Creiamo il modello

In [63]:
m = gp.Model('shortestPath')

Aggiungiamo le variabili e definiamo la funzione obiettivo

In [64]:
var_idxs = np.nonzero(adj_mat)
var_idxs = [(i, j) for i, j in zip(var_idxs[0], var_idxs[1])]
costs = {idx: adj_mat[idx] for idx in var_idxs}

arcs = m.addVars(var_idxs, lb = 0, ub = 1, obj=costs, name="arcs")
m.update()

In [65]:
m.getObjective()

<gurobi.LinExpr: 19.0 arcs[0,4] + 2.0 arcs[0,69] + 14.0 arcs[0,105] + 9.0 arcs[0,163] + 15.0 arcs[1,2] + 12.0 arcs[1,48] + 3.0 arcs[1,143] + 11.0 arcs[2,32] + 13.0 arcs[2,61] + 10.0 arcs[2,128] + 12.0 arcs[2,161] + 12.0 arcs[3,16] + 10.0 arcs[3,54] + 17.0 arcs[3,76] + 14.0 arcs[4,47] + 8.0 arcs[4,66] + 9.0 arcs[4,92] + 7.0 arcs[4,144] + 4.0 arcs[4,173] + 10.0 arcs[4,198] + 17.0 arcs[5,50] + 9.0 arcs[6,110] + 9.0 arcs[6,113] + 4.0 arcs[7,94] + 6.0 arcs[7,132] + 18.0 arcs[7,149] + 3.0 arcs[7,153] + 5.0 arcs[7,173] + 18.0 arcs[8,5] + 16.0 arcs[8,56] + 20.0 arcs[8,161] + 6.0 arcs[9,29] + 17.0 arcs[9,148] + 11.0 arcs[9,160] + 7.0 arcs[10,73] + 3.0 arcs[10,102] + 17.0 arcs[10,136] + 16.0 arcs[11,98] + 12.0 arcs[12,73] + 2.0 arcs[12,159] + 19.0 arcs[12,171] + 17.0 arcs[13,5] + 9.0 arcs[13,104] + arcs[13,113] + 2.0 arcs[13,115] + 10.0 arcs[13,117] + 18.0 arcs[13,126] + 5.0 arcs[13,172] + 17.0 arcs[13,176] + 18.0 arcs[14,63] + 12.0 arcs[14,119] + 11.0 arcs[14,135] + 9.0 arcs[14,142] + 14.0 arcs

Aggiungiamo i vincoli

In [66]:
b = np.zeros(n_nodes)
b[s] = 1
b[t] = -1

ct = m.addConstrs((arcs.sum('*', v) + b[v] == arcs.sum(v, '*') for v in range(n_nodes)))

Risolviamo il problema e stampiamo la soluzione trovata

In [67]:
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.4.0 25E246)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 200 rows, 760 columns and 1520 nonzeros (Min)
Model fingerprint: 0x31ac7dde
Model has 760 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 19 rows and 51 columns
Presolve time: 0.00s
Presolved: 181 rows, 709 columns, 1418 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -8.0000000e+00   4.000000e+00   0.000000e+00      0s
       9    1.9000000e+01   0.000000e+00   0.000000e+00      0s

Solved in 9 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.900000000e+01


In [68]:
print(f"Shortest path cost: {m.objVal:.4f}")
for v in var_idxs:
  if arcs[v].getAttr('X') == 1:
    print(f"{v[0]:3d} -> {v[1]:3d}: {arcs[v].getAttr('Obj'):.4f}")

Shortest path cost: 19.0000
  0 -> 105: 14.0000
105 -> 197: 5.0000


# Misc Utili

In [69]:
a = [1,2,3,4]
b = ["a", "b", "c", "d"]

## Itertools
Implementa molte funzioni utili per iterare su collezioni di elementi

In [70]:
import itertools

In [71]:
print(list(itertools.chain(a, b)))

[1, 2, 3, 4, 'a', 'b', 'c', 'd']


In [72]:
print(list(itertools.product(a,b)))

[(1, 'a'), (1, 'b'), (1, 'c'), (1, 'd'), (2, 'a'), (2, 'b'), (2, 'c'), (2, 'd'), (3, 'a'), (3, 'b'), (3, 'c'), (3, 'd'), (4, 'a'), (4, 'b'), (4, 'c'), (4, 'd')]


## zip, enumerate

In [73]:
for a_i, b_i in zip(a, b):
  print(a_i, b_i)

1 a
2 b
3 c
4 d


In [74]:
for x in zip(a, b):
  print(x)

(1, 'a')
(2, 'b')
(3, 'c')
(4, 'd')


In [75]:
for i, a_i in enumerate(a):
  print(i, a_i)

0 1
1 2
2 3
3 4


In [76]:
for x in enumerate(a):
  print(x)

(0, 1)
(1, 2)
(2, 3)
(3, 4)


# Esercizio

Un polo industriale produce televisori utilizzando tre linee di montaggio (A, B e C) per televisori piatti, curvi e monitor ad alte prestazioni. Per stabilire la produzione per il prossimo semestre e’ necessario considerare I seguenti aspetti:

  * Dalla vendita dei tre prodotti l’azienda ricava un profitto pari a 200, 400 e 500 euro rispettivamente.

  * Nel corso dei sei mesi, la linea A potra’ essere impengata per un totale di 150 ore al mese, la linea B per 130 al mese e la linea C per 170 ore al mese. Tuttavia, esclusivamente per il quarto mese di produzione sarà possibile usufruire di ulteriori 20 ore di lavoro sulla linea A e di 10 ore in meno sulla linea C.

  * Produrre un televisore a schermo piatto richiede 18 ore sulla linea A e 7 sulla B, un televisore curvo invece ne richiede 12 sulla linea B e 25 sulla C, mentre un monitor ad alte prestazioni richiede 10 ore di lavorazione sulla linea A, 9 sulla linea B e 14 sulla linea C.

  * In un impianto separato l’azienda esegue verifiche sui prodotti ed ha una capacita’ lavorativa mensile pari a 120 ore. Per eseguire tutte le verifiche necessarie sono richieste 5 ore per un televisore a schermo piatto, 8 per uno curvo e 9 per un monitor.

  * A seguito di analisi di mercato vi sono I seguenti vincoli sulla produzione: nel secondo mese la produzione di monitor deve supereare le 10 unita’, nel terzo e quinto mese la produzione di televisori curvi deve essere almeno pari al 30% di quella di televisori piatti ed infine nel sesto mese di produzione non e’ prevista richiesta di monitor ad alte prestazioni.

  * Temendo uno sciopero dei lavoratori, la direzione considera che nel primo mese di produzione la capacita’ lavorativa dell’impianto di verifica dei prodotti sara’ ridotta al 75%.

Sulla base di tali considerazioni, il problema e’ pianificiare la produzione del prossimo semestre con l’obiettivo di massimizzare il profitto dell’azienda.

In [77]:
#TODO